In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../dataset/train.csv')


In [2]:
df.columns

Index(['sample_id', 'catalog_content', 'image_link', 'price'], dtype='object')

In [3]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75000 entries, 0 to 74999
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   sample_id        75000 non-null  int64  
 1   catalog_content  75000 non-null  object 
 2   image_link       75000 non-null  object 
 3   price            75000 non-null  float64
dtypes: float64(1), int64(1), object(2)
memory usage: 2.3+ MB


,sample_id,price
count,75000.000000,75000.000000
mean,149841.917707,23.647654
std,86585.346513,33.376932
min,0.000000,0.130000
25%,73845.750000,6.795000
50%,150129.000000,14.000000
75%,225040.250000,28.625000
max,299438.000000,2796.000000


In [ ]:
def find_quantity(s: str)->float:

    """function to return the quantity of the item present"""
    # CRITICAL FIX: The logic here is fragile. Using .rfind() for quantity and unit extraction 
    # assumes a strict format 'value: X unit: Y'. We need to make sure the value 
    # is extracted correctly and converted to float, or return np.nan on failure.
    s = s.lower()
    value_start_marker = 'value: '
    unit_start_marker = 'unit:'
    
    start_idx = s.rfind(value_start_marker)
    end_idx = s.rfind(unit_start_marker)
    
    if start_idx != -1 and end_idx != -1 and start_idx < end_idx:
        value_string = s[start_idx + len(value_start_marker): end_idx].strip()
        try:
            return float(value_string)
        except ValueError:
            return np.nan
    return np.nan

def find_unit(s: str)-> str:

    """function to return the unit in which the item is carried/shipped"""
    s = s.lower()
    start_marker = 'unit: '
    start_index = s.rfind(start_marker)
    
    if start_index != -1:
        start_index += len(start_marker)
        
        end_index = s.find(' ', start_index)
        if end_index == -1:
            end_index = s.find("\n", start_index)
            
        unit_string = s[start_index: end_index] if end_index != -1 else s[start_index:]
        return unit_string.strip()
    return 'ambiguous'

def standardize_units(df: pd.DataFrame, col_name:str = 'unit'):

    unit_map = { # Volume Standards
        'fl oz': 'fl_oz', 'fluid ounce': 'fl_oz', 'fluid ounces': 'fl_oz', 
        'fl oz ': 'fl_oz', 'fl. oz': 'fl_oz', 'fl. oz.': 'fl_oz', 
        'fl.oz': 'fl_oz', 'fl ounce': 'fl_oz', 'fluid ounce(s)': 'fl_oz',
        'liters': 'ml', 'ltr': 'ml', 'ml': 'ml', 'millilitre': 'ml', 
        'milliliter': 'ml', 'mililitro': 'ml',
        
        # Weight Standards
        'ounce': 'oz', 'ounces': 'oz', 'oz': 'oz', 'ounce ': 'oz', 
        'pound': 'lb', 'pounds': 'lb', 'lb': 'lb', 'gram': 'g', 
        'grams': 'g', 'gramm': 'g', 'gr': 'g', 'grams(gm)': 'g', 
        'kg': 'kg', '7,2 oz': 'oz',

        # Count Standards 
        'count': 'count', 'ct': 'count', 'each': 'count', 'units': 'count', 
        'unità': 'count', 'tea bags': 'count', 'k-cups': 'count', 
        'capsule': 'count', 'paper cupcake liners': 'count', 
        
        # Count Standards 
        'pack': 'package', 'packs': 'package', 'box': 'package', 'jar': 'package', 
        'bottle': 'package', 'can': 'package', 'piece': 'package', 'pouch': 'package', 
        'bucket': 'package', 'bottles': 'package', 'bag': 'package', 
        'ziplock bags': 'package', 'carton': 'package', 'box/12': 'package',
        'packs':'package', 'boxes':'package', 
        
        # Ambiguous/Context
        'none': 'ambiguous', '-': 'ambiguous', '': 'ambiguous', '---': 'ambiguous', 
        'in': 'ambiguous', '24': 'ambiguous', '8': 'ambiguous', '1': 'ambiguous',
        'per box': 'context', 'per package': 'context', 'per carton': 'context', 
        'product_weight': 'context',
        
        # Area
        'sq ft': 'area', 'foot': 'area'
    }
        # Clean the raw column
    df['clean_unit'] = df['unit'].astype(str).str.strip().str.lower() # Ensure string conversion and lowercasing
    
    #standardization mapping
    df['standardized_unit'] = df['clean_unit'].replace(unit_map)
    
    category_map = {
        'fl_oz': 'Volume', 'ml': 'Volume',
        'oz': 'Weight', 'g': 'Weight', 'lb': 'Weight', 'kg': 'Weight', 
        'count': 'Discrete_Count',
        'package': 'Packaging_Count',
        'area': 'Area',
        'ambiguous': 'Ambiguous', 'context': 'Context', 
    }
    
    df['unit_category'] = df['standardized_unit'].replace(category_map)
    
    df['unit_category'] = df['unit_category'].fillna('Ambiguous')
    
    df = df.drop(columns=['clean_unit'])
    return df

# conversion constants
G_TO_OZ = 0.0352
LB_TO_OZ = 16.0
KG_TO_OZ = 35.274
ML_TO_FL_OZ = 0.0338

def normalize_quantities(df: pd.DataFrame):

    """
    convert raw quantity and standardized unit columns into total_normalized_quantity 
    and apply log transformation.
    """
    df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')
    df['total_normalized_quantity'] = np.nan
    
    # conditions and choices
    conditions = [
        # weight conversions normalize to oz
        (df['standardized_unit'] == 'oz'),
        (df['standardized_unit'] == 'g'),
        (df['standardized_unit'] == 'lb'),
        (df['standardized_unit'] == 'kg'),
        
        # volume conversions normalize to fl_oz
        (df['standardized_unit'] == 'fl_oz'),
        (df['standardized_unit'] == 'ml'), 
        
        # discrete count keep raw number
        (df['unit_category'].isin(['Discrete_Count', 'Packaging_Count']))
    ]
    
    choices = [
        # Weight
        df['quantity'],                         # oz -> oz
        df['quantity'] * G_TO_OZ,               # g -> oz
        df['quantity'] * LB_TO_OZ,              # lb -> oz
        df['quantity'] * KG_TO_OZ,              # kg -> oz
        
        # Volume
        df['quantity'],                         # fl_oz -> fl_oz
        df['quantity'] * ML_TO_FL_OZ,           # ml -> fl_oz
        
        # Count / Packaging
        df['quantity']                          # count/package 
    ]
    
    df['total_normalized_quantity'] = np.select(conditions, choices, default=np.nan)

    # Log transformation for Gradient Boosting stability
    df['log_normalized_quantity'] = np.log10(df['total_normalized_quantity'])
    
    return df

In [5]:
df['quantity'] = df['catalog_content'].apply(find_quantity)



In [ ]:
df.iloc[0]
df.describe()

df['unit'] = df['catalog_content'].apply(find_unit)
df = standardize_units(df)

df = normalize_quantities(df)


c:\Users\Madhav\GithubRepos\AmazonPricePredictor\Lib\site-packages\pandas-2.3.3-py3.11-win-amd64.egg\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\Madhav\GithubRepos\AmazonPricePredictor\Lib\site-packages\pandas-2.3.3-py3.11-win-amd64.egg\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,sample_id,price,quantity,total_normalized_quantity,log_normalized_quantity
count,75000.000000,75000.000000,74060.000000,62696.000000,6.269600e+04
mean,149841.917707,23.647654,54.306073,43.651875,-inf
std,86585.346513,33.376932,461.818743,284.545883,NaN
min,0.000000,0.130000,0.000000,0.000000,-inf
25%,73845.750000,6.795000,6.000000,5.700000,7.558749e-01
50%,150129.000000,14.000000,16.000000,16.000000,1.204120e+00
75%,225040.250000,28.625000,48.000000,42.000000,1.623249e+00
max,299438.000000,2796.000000,63882.000000,46752.000000,4.669800e+00


In [7]:
df['catalog_content'] = df['catalog_content'].str.lower()

has_value = df['catalog_content'].str.contains('value:', na = False)
has_unit = df['catalog_content'].str.contains('unit:', na = False)

print(has_value.sum())
print(has_unit.sum())

75000
75000


In [ ]:
df

In [1]:


def flatten_to_hierarchy(d, parent_key='', sep='/'):
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            # Recursively flatten
            items.extend(flatten_to_hierarchy(v, new_key, sep=sep).items())
        else:
            items.append((new_key, v))
    return dict(items)

data = {
    'numerical_columns': {
        "price": {
            "p_value": 0.05,
            "mean": 10,
            "drift_detected": True
        }
    }
}

flat_metrics = flatten_to_hierarchy(data)

print(flat_metrics)

{'numerical_columns/price/p_value': 0.05, 'numerical_columns/price/mean': 10, 'numerical_columns/price/drift_detected': True}
